# Однослойный перцептрон
Демонстрация с бесконечностью

In [ ]:
import numpy as np

# Функция активации нейрона в выходном слое и ее производная
def sigmoid(x):
    return 1 / (1 + np.exp(-x))
def dsigmoid(y):
    return y * (1 - y)

# Функция активации нейронов во внутреннем слое и ее производная
def tanh(x):
    return np.tanh(x)
def dtanh(y):
    return 1 - y * y

In [ ]:
from matplotlib import pyplot as plt

class Perceptron:
    def __init__(self, X, Y, hidden_size):
        
        # Пусть n - количество признаков, m - количество объектов, k - размерность результирующего вектора
        # Тогда X = X(n * m), Y = Y(k, m)
        assert X.shape[1] == Y.shape[1]
        self.X = X
        self.Y = Y
        self.m = X.shape[1]
        
        # Инициализируем матрицы весов рандомными числами (W1 - для скрытого слоя, W2 - для выходного, в нем всего 1 нейрон)
        self.W1 = np.random.randn(hidden_size, X.shape[0])
        self.W2 = np.random.randn(Y.shape[0], hidden_size)
        
        # Заполняем свободные векторы нулями
        self.b1 = np.zeros((hidden_size, 1))
        self.b2 = np.zeros((Y.shape[0], 1))
        
        # Здесь будем накапливать значения функции потерь на каждой итерации
        self.ce_loss = []
        self.mse_loss = []
        
    def learn(self, iter_num, learning_rate):
        
        # Итерируемся необходимое количество раз
        for i in range(iter_num):
            
            # Расчет взыешенных сумм каждого из нейронов, применение функций активации
            # Для того, чтобы заменить функции активации, необходимо заменить функцию при вычислении А1/А2
            Z1 = np.dot(self.W1, self.X) + self.b1                                     # (hidden_size, X.shape[1])
            A1 = tanh(Z1)
            Z2 = np.dot(self.W2, A1) + self.b2                                         # (Y.shape[0], X.shape[1] = Y.shape[1])
            A2 = sigmoid(Z2)

            # Среднеквадратичная (MSE) функция потерь
            mse_loss = np.sum((A2 - self.Y) ** 2) / self.m
            
            # Cross entropy функция потерь
            eps = 1e-8
            logprobs = self.Y * np.log(A2 + eps) + (1 - self.Y) * np.log(1 - A2 + eps)
            ce_loss = (-1 / self.m) * np.sum(logprobs)
            
            # Запоминание значения функции потерь и печать на каждой 1000-й итерации
            self.mse_loss.append(mse_loss)
            self.ce_loss.append(ce_loss)
            if i % 1000 == 0: print(f"MSE loss: {mse_loss}, CE loss: {ce_loss}")

            # Алгоритм обратного распространения ошибки
            # Производные высчитываются по правилу производной для сложной функции
            # Оптимизируется среднеквадратичная функция потерь
            # Для оптимизации cross entropy необходимо заменить производную функции потерь по A2 - self.Y
            # Для этого необходимо найти dError/dW
            # При замене функций активации необходимо также заменить соответствующую ей производную
            dA2 = (A2 - self.Y) * dsigmoid(A2)        # dLoss/dA * dSig/dZ2            # (Y.shape[0], Y.shape[1] = X.shape[1])
            dW2 = np.dot(dA2, A1.T)                   # dZ2/dW2                        # (Y.shape[0], hidden_size)
            db2 = np.sum(dA2, axis=1, keepdims=True)                                   # (Y.shape[0], 1)
            dA1 = np.dot(self.W2.T, dA2) * dtanh(A1)  # dA2 * dZ2/dA1 * dTanh/dZ       # (hidden_size, Y.shape[1] = X.shape[1])
            dW1 = np.dot(dA1, self.X.T)               # dZ1/dW                         # (hidden_size, X.shape[0])
            db1 = np.sum(dA1, axis=1, keepdims=True)                                   # (hidden_size, 1)
            
            # Обновление весов методом градиентного спуска
            self.W2 -= learning_rate * dW2
            self.b2 -= learning_rate * db2
            self.W1 -= learning_rate * dW1
            self.b1 -= learning_rate * db1
            
    def predict(self, X):
        # Используем найденные матрицы весов
        A1 = tanh(np.dot(self.W1, X) + self.b1)
        A2 = sigmoid(np.dot(self.W2, A1) + self.b2)
        return (A2[0] > 0.5).astype(int)
    
    def plot(self):
        plt.figure(figsize=(10,3))
        plt.subplot(1, 2, 1)
        plt.ylabel('MSE loss')
        plt.xlabel('Iterations')
        plt.plot(self.mse_loss)
        plt.subplot(1, 2, 2)
        plt.ylabel('CE loss')
        plt.xlabel('Iterations')
        plt.plot(self.ce_loss)

## Разделение бесконечности

In [ ]:
def create_infinity(m, noise=0.1):
    D = 2
    N = m//2 # количество точек для каждого класса
    X = np.zeros((m, D))
    Y = np.zeros((m, 1), dtype='uint8')

    # Class 0 (top part))
    t = np. linspace(0, 1, N)
    x_top = t*2 - 1 # scale to [-1, 1]
    y_top = np.sin(2*np.pi*t) + np.random.randn(N)*noise

    X[0:N] = np.c_[x_top, y_top]
    Y[0:N] = 0

    # Class 1 (bottom part)
    x_bottom = t*2 - 1  # scale to [-1, 1]
    y_bottom = -np.sin(2*np.pi*t) + np.random.randn(N)*noise

    X[N:m] = np.c_[x_bottom, y_bottom]
    Y[N:m] = 1

    return X.T, Y.T

In [ ]:
X, Y = create_infinity(400)
plt.scatter(X[0, :], X[1, :], c=Y)

In [ ]:
# Строим разделяющую  прямую, показывающую результат обучения нейросети
def plot_division(perceptron):
    x_min, x_max = X[0, :].min() - 1, X[0, :].max() + 1
    y_min, y_max = X[1, :].min() - 1, X[1, :].max() + 1
    h = 0.01
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = perceptron.predict(np.c_[xx.ravel(), yy.ravel()].T)
    Z = Z.reshape(xx.shape)
    plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral)
    plt.ylabel('x2')
    plt.xlabel('x1')
    plt.scatter(X[0, :], X[1, :], c=Y, cmap=plt.cm.Spectral);

### Один нейрон

In [ ]:
perceptron = Perceptron(X, Y, 1)
perceptron.learn(10000, 0.01)

In [ ]:
perceptron.plot()

In [ ]:
# Строим разделяющую  прямую, показывающую результат обучения нейросети
plot_division(perceptron)

### Четыре нейрона

In [ ]:
perceptron = Perceptron(X, Y, 4)
perceptron.learn(10000, 0.01)

In [ ]:
perceptron.plot()

In [ ]:
plot_division(perceptron)

### много нейронов - стало ли лучше?

In [ ]:
perceptron = Perceptron(X, Y, 100)
perceptron.learn(10000, 0.01)

In [ ]:
perceptron.plot()

In [ ]:
plot_division(perceptron)

---

**МОЯ РАБОТА**

---

Визуально я как минимум разделил бы поле на 4 области, минимум 2 нейрона, для макисмума добавил бы на каждую точку пересечения 2 нейрона, точек 3, тогда идейно 2-8 нейронов.

In [ ]:
Q=2
print("Q =", Q)
perceptron = Perceptron(X, Y, Q)
perceptron.learn(10000, 0.01)
perceptron.plot()
print()

In [ ]:
plot_division(perceptron)

In [ ]:
Q=4
print("Q =", Q)
perceptron = Perceptron(X, Y, Q)
perceptron.learn(10000, 0.01)
perceptron.plot()
print()

In [ ]:
plot_division(perceptron)

In [ ]:
Q=6
print("Q =", Q)
perceptron = Perceptron(X, Y, Q)
perceptron.learn(10000, 0.01)
perceptron.plot()
print()

In [ ]:
plot_division(perceptron)

In [ ]:
Q=8
print("Q =", Q)
perceptron = Perceptron(X, Y, Q)
perceptron.learn(10000, 0.01)
perceptron.plot()
print()

In [ ]:
plot_division(perceptron)

Вывод на 5 уже избыточная область справа сверху красная, тогда смотрим ещё раз 2-4

In [ ]:
Q=2
print("Q =", Q)
perceptron = Perceptron(X, Y, Q)
perceptron.learn(10000, 0.01)
perceptron.plot()
print()

In [ ]:
plot_division(perceptron)

In [ ]:
Q=3
print("Q =", Q)
perceptron = Perceptron(X, Y, Q)
perceptron.learn(10000, 0.01)
perceptron.plot()
print()

In [ ]:
plot_division(perceptron)

In [ ]:
Q=4
print("Q =", Q)
perceptron = Perceptron(X, Y, Q)
perceptron.learn(10000, 0.01)
perceptron.plot()
print()

In [ ]:
plot_division(perceptron)

**Вывод: минимальная, стабильно удовлетворяюще разделяющая - при 3 нейронах**

In [ ]:
def calculate_accuracy(perceptron, X, Y):
    predictions = perceptron.predict(X)
    accuracy = np.mean(predictions == Y[0])
    return accuracy

class SimpleEnhancedPerceptron(Perceptron):
    def __init__(self, X, Y, hidden_size):
        super().__init__(X, Y, hidden_size)
        self.accuracies = []
        
    def learn(self, iter_num, learning_rate):
        for i in range(iter_num):
            Z1 = np.dot(self.W1, self.X) + self.b1
            A1 = tanh(Z1)
            Z2 = np.dot(self.W2, A1) + self.b2
            A2 = sigmoid(Z2)

            mse_loss = np.sum((A2 - self.Y) ** 2) / self.m
            eps = 1e-8
            logprobs = self.Y * np.log(A2 + eps) + (1 - self.Y) * np.log(1 - A2 + eps)
            ce_loss = (-1 / self.m) * np.sum(logprobs)
            
            self.mse_loss.append(mse_loss)
            self.ce_loss.append(ce_loss)
            
            if i % 500 == 0:
                accuracy = calculate_accuracy(self, self.X, self.Y)
                self.accuracies.append(accuracy)
                if i % 1000 == 0: 
                    print(f"Iter {i}: CE loss: {ce_loss:.4f}, Accuracy: {accuracy:.4f}")

            dA2 = (A2 - self.Y) * dsigmoid(A2)
            dW2 = np.dot(dA2, A1.T)
            db2 = np.sum(dA2, axis=1, keepdims=True)
            dA1 = np.dot(self.W2.T, dA2) * dtanh(A1)
            dW1 = np.dot(dA1, self.X.T)
            db1 = np.sum(dA1, axis=1, keepdims=True)
            
            self.W2 -= learning_rate * dW2
            self.b2 -= learning_rate * db2
            self.W1 -= learning_rate * dW1
            self.b1 -= learning_rate * db1

    def plot_enhanced(self):
        plt.figure(figsize=(12, 4))
        
        plt.subplot(1, 2, 1)
        plt.plot(self.ce_loss)
        plt.title('Функция потерь (CE)')
        plt.xlabel('Итерации')
        plt.ylabel('Loss')
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 2, 2)
        iterations = range(0, len(self.accuracies) * 500, 500)
        plt.plot(iterations, self.accuracies, 'r-')
        plt.title('Точность на обучающих данных')
        plt.xlabel('Итерации')
        plt.ylabel('Accuracy')
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()

In [ ]:
neuron_counts = [1, 2, 3, 4, 5, 16]

for hidden_size in neuron_counts:
    print(f"\n=== Тестируем {hidden_size} нейронов ===")
    perceptron = SimpleEnhancedPerceptron(X, Y, hidden_size)
    perceptron.learn(8000, 0.01)
    
    perceptron.plot_enhanced()
    plt.suptitle(f'Результаты для {hidden_size} нейронов')
    plt.show()
    
    plot_division(perceptron)
    plt.title(f'Разделяющая граница ({hidden_size} нейронов)')
    plt.show()

**Видим результаты для: 1 - явно недоучено, 16 явно переобучено, 2 3 4 5 - видно что лучшая точность достигается на 3 и потом не особо улучшается**

In [ ]:
X_test, Y_test = create_infinity(200, noise=0.1)

print("=== Сравнение моделей ===")

print("1. Модель с 3 нейронами:")
perceptron_3 = Perceptron(X, Y, 3)
perceptron_3.learn(8000, 0.01)
train_acc_3 = np.mean(perceptron_3.predict(X) == Y[0])
test_acc_3 = np.mean(perceptron_3.predict(X_test) == Y_test[0])
print(f"   Точность на обучающих: {train_acc_3:.4f}")
print(f"   Точность на тестовых:  {test_acc_3:.4f}")

print("\n2. Модель с 63 нейронами:")
perceptron_63 = Perceptron(X, Y, 63)
perceptron_63.learn(8000, 0.01)
train_acc_63 = np.mean(perceptron_63.predict(X) == Y[0])
test_acc_63 = np.mean(perceptron_63.predict(X_test) == Y_test[0])
print(f"   Точность на обучающих: {train_acc_63:.4f}")
print(f"   Точность на тестовых:  {test_acc_63:.4f}")

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(perceptron_3.ce_loss, label='3 нейрона')
plt.plot(perceptron_63.ce_loss, label='63 нейрона')
plt.title('Сравнение потерь')
plt.xlabel('Итерации')
plt.ylabel('CE Loss')
plt.legend()

plt.subplot(1, 2, 2)
categories = ['3 нейрона', '63 нейрона']
train_accs = [train_acc_3, train_acc_63]
test_accs = [test_acc_3, test_acc_63]
x = range(len(categories))
plt.bar(x, train_accs, width=0.4, label='Обучающие', alpha=0.7)
plt.bar([i + 0.4 for i in x], test_accs, width=0.4, label='Тестовые', alpha=0.7)
plt.xticks([i + 0.2 for i in x], categories)
plt.title('Сравнение точности')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

**Вывод: точность уменьшена, значит переобучается, но на таком достаточно простом разделении это плохо видно**